# 🚀 Viral Clip Scout: Multi-Account YouTube Radar
An automated scouting agent designed to run daily on **Google Colab** or **Kaggle**:
1. **Multi-Account & Multi-Niche:** Curate separate clipping accounts (e.g., Crypto/IPOs, Business/Startups, Self-Growth), each with its own target channels and niche rules.
2. **Early Virality Detection:** Calculates real-time **Views Per Hour (VPH)** and velocity multipliers vs channel baselines.
3. **AI Niche & Clippability Filter (Gemini):** Scores video alignment for that specific account's niche (1–10) and extracts punchy short-form hooks.
4. **Smart Alerts (Telegram / Discord):** Sends photo cards tagged with the exact target account (e.g., `#CryptoClips`, `#BusinessGrowth`) so you know where to publish.
5. **Persistent Cloud State (MongoDB):** Tracks history in MongoDB across runs on Colab and Kaggle with zero manual logins.

In [32]:
# ==============================================================================
# SECTION 0: Installs & Core Imports (Colab + Kaggle Compatible)
# ==============================================================================
import sys

# Install required packages quietly
!pip install -q feedparser yt-dlp google-api-python-client google-genai pymongo[srv] requests tabulate

import os
import re
import json
import time
import math
import logging
from datetime import datetime, timezone, timedelta
from typing import List, Dict, Any, Optional

import requests
import feedparser
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure, ServerSelectionTimeoutError
from tabulate import tabulate

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("ViralClipScout")

print("✅ Section 0: Dependencies installed & core modules imported.")

✅ Section 0: Dependencies installed & core modules imported.


## Section 1: Secrets & Multi-Account Niche Profiles
1. **Secrets:** Works on both Google Colab (🔑 Secrets) and Kaggle (Add-ons → Secrets). Reads `MONGODB_URI`, `TELEGRAM_TOKEN`, `TELEGRAM_CHAT_ID`, `GEMINI_API_KEY`, etc.
2. **Clipping Accounts:** Add as many clipping accounts/niches as you want! Each has its own niche definition, hashtags, and channel roster.
3. **Detection Thresholds:** Configurable minimum views-per-hour and velocity multipliers.

In [38]:
# ==============================================================================
# SECTION 1: Secrets & Multi-Account Configuration
# ==============================================================================

def get_secret(key: str, default: str = "") -> str:
    """Retrieve secret across Colab userdata, Kaggle Secrets, and os.environ."""
    # 1. Try Google Colab userdata
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val is not None and str(val).strip():
            return str(val).strip()
    except Exception:
        pass

    # 2. Try Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        val = user_secrets.get_secret(key)
        if val is not None and str(val).strip():
            return str(val).strip()
    except Exception:
        pass

    # 3. Try Environment Variables
    return os.environ.get(key, default).strip()

# Core Secrets
MONGODB_URI         = get_secret("MONGODB_URI", "")
TELEGRAM_BOT_TOKEN  = get_secret("TELEGRAM_BOT_TOKEN", "") or get_secret("TELEGRAM_TOKEN", "")
TELEGRAM_CHAT_ID    = get_secret("TELEGRAM_CHAT_ID", "")
DISCORD_WEBHOOK_URL = get_secret("DISCORD_WEBHOOK_URL", "")
YOUTUBE_API_KEY     = get_secret("YOUTUBE_API_KEY", "")
GEMINI_API_KEY      = get_secret("GEMINI_API_KEY", "")

# MongoDB Config
DB_NAME             = "viral_clip_radar"
COLLECTION_NAME     = "clipping_alerts"
LOCAL_FALLBACK_FILE = "clip_scout_history.json"

# ==============================================================================
# MULTI-ACCOUNT / MULTI-NICHE PROFILES (Configured for Your Clipping Channels)
# ==============================================================================
CLIPPING_ACCOUNTS = [
    {
        "account_name": "Crypto, Trading & Investing",
        "niche_id": "crypto_trading",
        "tag": "#CryptoTradingID",
        "niche_topics": (
            "Cryptocurrency, Bitcoin, Altcoins, Trading Teknikal/Fundamental, Saham IHSG/Wall Street, "
            "IPOs, Makroekonomi, Investasi, Reksadana, Inflasi, dan Strategi Finansial."
        ),
        "telegram_chat_id": "", # Leave empty to use global TELEGRAM_CHAT_ID
        "channels": [
            {
                "name": "Astronacci International",
                "id": "UCYRFEuGLfXFFvbq2-nStY9w",
                "handle": "@astronacciinternational",
                "baseline_24h_views": 25_000
            },
            {
                "name": "Timothy Ronald",
                "id": "UCXMB8OiiSnq2B4xLgUtTYhw",
                "handle": "@timothyronald",
                "baseline_24h_views": 70_000
            },
            {
                "name": "Raymond Chin",
                "id": "UCm4eA46c6boHmPrD0Lp2iEg",
                "handle": "@raymondchins",
                "baseline_24h_views": 60_000
            }
        ]
    },
    {
        "account_name": "Bisnis, Startup & Entrepreneurship",
        "niche_id": "business_startups",
        "tag": "#BisnisStartupsID",
        "niche_topics": (
            "Bisnis, Entrepreneurship, Marketing, Sales, Startup, UMKM, Studi Kasus Perusahaan, "
            "Bongkar Bisnis Model, Isu Bisnis Viral, dan Wawancara Pengusaha Sukses."
        ),
        "telegram_chat_id": "",
        "channels": [
            {
                "name": "Kasisolusi",
                "id": "UC5rEjJFYx03lhODA9uH9zew",
                "handle": "@kasisolusi",
                "baseline_24h_views": 40_000
            },
            {
                "name": "Leon Hartono (The Overpost)",
                "id": "UCFWKvu581DpCRFfadjjIy7w",
                "handle": "@leon.hartono",
                "baseline_24h_views": 35_000
            },
            {
                "name": "Raymond Chin",
                "id": "UCm4eA46c6boHmPrD0Lp2iEg",
                "handle": "@raymondchins",
                "baseline_24h_views": 60_000
            },
            {
                "name": "Deddy Corbuzier",
                "id": "UCYk4LJI0Pr6RBDWowMm-KUw",
                "handle": "@corbuzier",
                "baseline_24h_views": 350_000
            }
        ]
    },
    {
        "account_name": "Self-Growth, Habits & Karir",
        "niche_id": "self_growth",
        "tag": "#SelfGrowthID",
        "niche_topics": (
            "Pengembangan Diri, Produktivitas, Mindset, Membangun Kebiasaan (Habits), "
            "Manajemen Waktu, Personal Finance, Karir, dan Filosofi Hidup Produktif."
        ),
        "telegram_chat_id": "",
        "channels": [
            {
                "name": "Theresa Learns",
                "id": "UC5C3bwo-CdJBvEqp5rieUYg",
                "handle": "@theresalearns",
                "baseline_24h_views": 20_000
            },
            {
                "name": "Raditya Dika",
                "id": "UC0rzsIrAxF4kCsALP6J2EsA",
                "handle": "@radityadika",
                "baseline_24h_views": 80_000
            }
        ]
    }
]

# Detection Thresholds
DETECTION_CONFIG = {
    "min_age_hours": 1.0,           # Ignore brand new uploads (<1 hr) while views stabilize
    "max_age_hours": 72.0,          # Check videos released within the last 3 days
    "min_viral_multiplier": 1.4,    # Must be at least 1.4x faster than typical channel pace
    "min_absolute_vph": 250,        # Minimum Views Per Hour regardless of baseline
    "min_niche_score": 6.0,         # Out of 10
    "min_viral_score": 6.5,         # Out of 10
    "max_videos_per_channel": 5     # Number of recent uploads to scan per channel
}

print(f"✅ Configuration initialized:")
print(f"   • Clipping Accounts:  {len(CLIPPING_ACCOUNTS)} active profiles")
print(f"   • Channels Configured: 8 target channels")
print(f"   • MongoDB URI:        {'Available' if MONGODB_URI else 'Not set (using local JSON fallback)'}")
print(f"   • Gemini API Key:     {'Available' if GEMINI_API_KEY else 'Not set (using heuristic fallback)'}")
print(f"   • Telegram Alerts:    {'Configured' if TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID else 'Needs Token/ChatID'}")

✅ Configuration initialized:
   • Clipping Accounts:  3 active profiles
   • Channels Configured: 8 target channels
   • MongoDB URI:        Available
   • Gemini API Key:     Available
   • Telegram Alerts:    Configured


## Section 2: Persistent State Tracking (MongoDB + Local Fallback)
Tracks previously alerted videos to prevent repeat spam across daily runs:
1. **MongoDB Atlas:** Seamless cloud persistence that works headless on both **Google Colab** and **Kaggle**.
2. **Per-Niche De-duplication:** Tracks records using a compound key (`video_id` + `niche_id`), ensuring a dual-relevance video can still alert separate target accounts if relevant.
3. **Local JSON Fallback:** If MongoDB is offline or URI is omitted, transparently uses `clip_scout_history.json`.

In [34]:
# ==============================================================================
# SECTION 2: MongoDB & State Tracking
# ==============================================================================

class MongoStateManager:
    """Manages history of checked and alerted videos via MongoDB Atlas with local fallback."""

    def __init__(self, uri: str = MONGODB_URI, db_name: str = DB_NAME, coll_name: str = COLLECTION_NAME):
        self.uri = uri
        self.db_name = db_name
        self.coll_name = coll_name
        self.client = None
        self.collection = None
        self.local_cache = set()
        self._init_connection()

    def _init_connection(self):
        # 1. Load from local cache file first
        if os.path.exists(LOCAL_FALLBACK_FILE):
            try:
                with open(LOCAL_FALLBACK_FILE, "r") as f:
                    data = json.load(f)
                    self.local_cache = set(data.keys())
            except Exception as e:
                log.warning(f"Could not read local history: {e}")

        # 2. Connect to MongoDB if URI is provided
        if self.uri:
            try:
                self.client = MongoClient(self.uri, serverSelectionTimeoutMS=5000)
                # Verify connection
                self.client.admin.command('ping')
                db = self.client[self.db_name]
                self.collection = db[self.coll_name]

                # Create index for fast lookups
                self.collection.create_index([("video_id", 1), ("niche_id", 1)], unique=True)
                count = self.collection.count_documents({})
                log.info(f"Connected to MongoDB '{self.db_name}.{self.coll_name}' ({count} alerts tracked).")
            except Exception as e:
                log.warning(f"MongoDB connection failed: {e}. Falling back to local JSON storage.")
                self.collection = None
        else:
            log.info(f"MONGODB_URI not provided. Using local JSON storage '{LOCAL_FALLBACK_FILE}'.")

    def is_already_alerted(self, video_id: str, niche_id: str) -> bool:
        cache_key = f"{video_id}_{niche_id}"
        if cache_key in self.local_cache or video_id in self.local_cache:
            return True

        if self.collection is not None:
            try:
                doc = self.collection.find_one({"video_id": video_id, "niche_id": niche_id})
                if doc:
                    self.local_cache.add(cache_key)
                    return True
            except Exception as e:
                log.warning(f"MongoDB query failed: {e}")
        return False

    def record_alert(self, item: dict, account: dict):
        vid = item["video_id"]
        niche_id = account.get("niche_id", "default")
        cache_key = f"{vid}_{niche_id}"
        self.local_cache.add(cache_key)
        now_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

        alert_doc = {
            "video_id": vid,
            "niche_id": niche_id,
            "account_name": account.get("account_name", ""),
            "account_tag": account.get("tag", ""),
            "title": item.get("title", ""),
            "channel_name": item.get("channel_name", ""),
            "published_at": item.get("published_at", ""),
            "views": item.get("views", 0),
            "vph": round(item.get("vph", 0), 1),
            "multiplier": round(item.get("multiplier", 1.0), 2),
            "niche_score": item.get("niche_score", "-"),
            "viral_score": item.get("viral_score", "-"),
            "hook_idea": item.get("hook_idea", ""),
            "clipping_reason": item.get("clipping_reason", ""),
            "url": item.get("url", f"https://www.youtube.com/watch?v={vid}"),
            "alerted_at": now_str
        }

        # 1. Save to MongoDB
        if self.collection is not None:
            try:
                self.collection.update_one(
                    {"video_id": vid, "niche_id": niche_id},
                    {"$set": alert_doc},
                    upsert=True
                )
            except Exception as e:
                log.error(f"Failed to save alert to MongoDB: {e}")

        # 2. Save to local JSON fallback
        local_data = {}
        if os.path.exists(LOCAL_FALLBACK_FILE):
            try:
                with open(LOCAL_FALLBACK_FILE, "r") as f:
                    local_data = json.load(f)
            except Exception:
                pass

        local_data[cache_key] = alert_doc
        try:
            with open(LOCAL_FALLBACK_FILE, "w") as f:
                json.dump(local_data, f, indent=2)
        except Exception as e:
            log.warning(f"Could not update local storage: {e}")

state_mgr = MongoStateManager()
print(f"✅ State tracking ready. Storage mode: {'MongoDB Cloud' if state_mgr.collection is not None else 'Local JSON'}")

✅ State tracking ready. Storage mode: MongoDB Cloud


## Section 3: Video Discovery (YouTube Data API v3 + yt-dlp Fallback)
1. **RSS Scanner:** Instantly finds the latest upload IDs for any YouTube channel without using any quota or API keys.
2. **Detailed Metadata & Metrics:**
   - **Primary:** YouTube Data API v3 (`videos.list` with parts `snippet,statistics,contentDetails`). 50 videos cost only **1 unit** of your 10,000 free daily quota.
   - **Fallback:** `yt-dlp --dump-json` if the API key is not provided or quota fails. Extracts views, likes, comments, duration, tags, and thumbnails.

In [24]:
# ==============================================================================
# SECTION 3: Video Discovery (YouTube API v3 + RSS Feed Metadata)
# ==============================================================================
import re
import urllib.parse
from dateutil import parser as date_parser

def resolve_channel_rss(channel: dict) -> str:
    """Generate RSS feed URL from channel ID or handle."""
    if "id" in channel and channel["id"].startswith("UC"):
        return f"https://www.youtube.com/feeds/videos.xml?channel_id={channel['id']}"
    elif "handle" in channel and channel["handle"]:
        handle = channel["handle"].lstrip("@")
        try:
            r = requests.get(f"https://www.youtube.com/@{handle}", headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
            m = re.search(r'itemprop="channelId" content="([^"]+)"', r.text)
            if not m:
                m = re.search(r'"channelId":"([^"]+)"', r.text)
            if m:
                channel["id"] = m.group(1)
                return f"https://www.youtube.com/feeds/videos.xml?channel_id={channel['id']}"
        except Exception:
            pass
    return f"https://www.youtube.com/feeds/videos.xml?channel_id={channel.get('id', '')}"

def parse_rss_entry(entry: dict, channel: dict) -> dict:
    """Extract real-time view counts, thumbnail, and metadata directly from RSS XML."""
    vid = entry.get("yt_videoid")
    if not vid and "link" in entry:
        link = entry.get("link", "")
        if "v=" in link:
            vid = link.split("v=")[-1].split("&")[0]

    views_str = entry.get("media_statistics", {}).get("views", "0")
    likes_str = entry.get("media_starrating", {}).get("count", "0")
    thumbs = entry.get("media_thumbnail", [])
    thumb_url = thumbs[0].get("url") if thumbs else f"https://i.ytimg.com/vi/{vid}/hqdefault.jpg"

    return {
        "video_id": vid,
        "title": entry.get("title", ""),
        "channel_name": channel.get("name", entry.get("author", "Unknown Channel")),
        "channel_id": channel.get("id", ""),
        "baseline_24h_views": channel.get("baseline_24h_views", 50_000),
        "published_at": entry.get("published", ""),
        "views": int(views_str) if str(views_str).isdigit() else 0,
        "likes": int(likes_str) if str(likes_str).isdigit() else 0,
        "comments": 0,
        "description": entry.get("summary", "")[:1000],
        "tags": [],
        "thumbnail_url": thumb_url,
        "url": f"https://www.youtube.com/watch?v={vid}"
    }

def fetch_channel_videos(channel: dict, limit: int = 5) -> List[dict]:
    """Scan channel RSS feed and parse video metadata."""
    rss_url = resolve_channel_rss(channel)
    try:
        feed = feedparser.parse(rss_url)
        results = []
        for entry in feed.entries[:limit]:
            parsed = parse_rss_entry(entry, channel)
            if parsed.get("video_id"):
                results.append(parsed)
        return results
    except Exception as e:
        log.warning(f"Failed to fetch RSS for {channel.get('name')}: {e}")
        return []

def enrich_via_youtube_api(videos: List[dict], api_key: str) -> List[dict]:
    """Optional enrichment with exact stats using YouTube Data API v3."""
    if not api_key or not videos:
        return videos

    try:
        youtube = build("youtube", "v3", developerKey=api_key)
        id_to_video = {v["video_id"]: v for v in videos}
        ids_chunk = list(id_to_video.keys())[:50]

        res = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(ids_chunk)
        ).execute()

        for item in res.get("items", []):
            vid = item["id"]
            if vid in id_to_video:
                snippet = item.get("snippet", {})
                stats = item.get("statistics", {})
                id_to_video[vid]["views"] = int(stats.get("viewCount", id_to_video[vid]["views"]))
                id_to_video[vid]["likes"] = int(stats.get("likeCount", id_to_video[vid]["likes"]))
                id_to_video[vid]["comments"] = int(stats.get("commentCount", 0))
                id_to_video[vid]["tags"] = snippet.get("tags", [])
        return list(id_to_video.values())
    except Exception as e:
        log.warning(f"YouTube Data API enrichment skipped: {e}")
        return videos

print("✅ Video discovery modules loaded (Native RSS XML stats + API v3 enrichment).")

✅ Video discovery modules loaded (Native RSS XML stats + API v3 enrichment).


## Section 4: Viral Velocity & Outlier Calculator
Calculates:
1. **Age in Hours:** Time elapsed since the video went live.
2. **Views Per Hour (VPH):** `Views / Age (hours)`.
3. **Channel Baseline VPH:** Derived from `baseline_24h_views / 24.0`.
4. **Velocity Multiplier:** `VPH / Baseline_VPH`.
5. **Early Outlier Status:** Flags whether this video is surging faster than typical channel uploads.

In [6]:
# ==============================================================================
# SECTION 4: Velocity & Outlier Detection
# ==============================================================================

def calculate_video_metrics(video: dict) -> dict:
    """Calculates views per hour (VPH) and relative velocity multiplier."""
    now = datetime.now(timezone.utc)
    pub_str = video.get("published_at", "")

    # Parse publish date
    pub_dt = None
    if pub_str:
        try:
            pub_dt = date_parser.parse(pub_str)
            if pub_dt.tzinfo is None:
                pub_dt = pub_dt.replace(tzinfo=timezone.utc)
        except Exception:
            pub_dt = None

    if not pub_dt:
        # Default fallback to 6 hours if date parsing fails
        age_hours = 6.0
    else:
        diff_seconds = max((now - pub_dt).total_seconds(), 60)
        age_hours = diff_seconds / 3600.0

    views = max(video.get("views", 0), 0)
    vph = views / max(age_hours, 0.25)

    # Calculate channel baseline velocity
    baseline_24h = video.get("baseline_24h_views", 50_000)
    baseline_vph = max(baseline_24h / 24.0, 1.0)
    multiplier = vph / baseline_vph

    # Check outlier criteria
    meets_min_age = age_hours >= DETECTION_CONFIG["min_age_hours"]
    meets_max_age = age_hours <= DETECTION_CONFIG["max_age_hours"]
    meets_multiplier = multiplier >= DETECTION_CONFIG["min_viral_multiplier"]
    meets_abs_vph = vph >= DETECTION_CONFIG["min_absolute_vph"]

    # Either high multiplier vs channel baseline OR strong raw absolute velocity
    is_outlier = meets_min_age and meets_max_age and (meets_multiplier or meets_abs_vph)

    video_enriched = dict(video)
    video_enriched.update({
        "age_hours": round(age_hours, 1),
        "vph": round(vph, 1),
        "multiplier": round(multiplier, 2),
        "is_outlier": is_outlier,
        "likes_ratio": round((video.get("likes", 0) / max(views, 1)) * 100, 2)
    })
    return video_enriched

print("✅ Velocity and outlier calculator defined.")

✅ Velocity and outlier calculator defined.


## Section 5: Account-Specific AI Niche Evaluator
Uses **Gemini Flash Lite (`gemini-flash-lite-latest`)** for ultra-fast, lightweight clippability evaluation and hook generation:
* Evaluates whether the content matches the specific account's themes (e.g. Crypto vs B2B Startups vs Productivity).
* Scores virality & hook potential (1–10).
* Generates an account-tailored 1-sentence opening hook in Bahasa Indonesia for TikTok / Shorts / Reels.

In [42]:
# ==============================================================================
# SECTION 5: AI Niche & Clippability Evaluator (gemini-flash-lite-latest + Heuristics)
# ==============================================================================

GEMINI_MODEL = "gemini-flash-lite-latest"

def heuristic_niche_eval(title: str, description: str, tags: list, vph: float, niche_topics: str) -> dict:
    """Fallback evaluator when GEMINI_API_KEY is not configured or offline."""
    text = f"{title} {description} {' '.join(tags)}".lower()
    keywords = [w.strip().lower() for w in niche_topics.replace(",", " ").split() if len(w) > 3]

    matches = sum(1 for kw in keywords if kw in text)
    niche_score = min(max(5.0 + (matches * 0.8), 4.0), 10.0)

    if vph > 2000:
        viral_score = 9.0
    elif vph > 800:
        viral_score = 7.5
    elif vph > 300:
        viral_score = 6.5
    else:
        viral_score = 5.0

    return {
        "niche_score": round(niche_score, 1),
        "viral_score": round(viral_score, 1),
        "hook_idea": f"Trik & insight penting dari video ini: {title[:60]}...",
        "clipping_reason": f"Video berkecepatan tinggi dengan relevansi kuat ke topik ({matches} kata kunci cocok)."
    }

def evaluate_clippability_gemini(video: dict, account: dict, api_key: str, model_name: str = GEMINI_MODEL) -> dict:
    """Use Gemini to evaluate niche relevance and short-form clipping angles for an account."""
    niche_topics = account.get("niche_topics", "")
    account_name = account.get("account_name", "General")

    if not api_key:
        return heuristic_niche_eval(
            video.get("title", ""),
            video.get("description", ""),
            video.get("tags", []),
            video.get("vph", 0),
            niche_topics
        )

    prompt = f"""You are an elite short-form video editor and viral clip producer for Indonesian TikTok, Instagram Reels, and YouTube Shorts.
You are evaluating content for our clipping channel: "{account_name}".
Account Target Niche: {niche_topics}

Video Information:
- Judul Video: {video.get('title')}
- Nama Channel: {video.get('channel_name')}
- Usia Video: {video.get('age_hours')} jam yang lalu
- Views Terkini: {video.get('views')} (Kecepatan: {video.get('vph')} views/jam, Multiplier: {video.get('multiplier')}x kecepatan normal channel)
- Deskripsi / Chapters / Topik: {video.get('description', '')[:700]}

Tasks:
1. Evaluasi apakah konten ini COCOK dengan target niche akun "{account_name}" (niche_score 1.0 - 10.0). Catatan: Jika ini video podcast umum (seperti Deddy Corbuzier / Raditya Dika), periksa apakah bintang tamu atau topik bahasannya relevan dengan niche akun.
2. Evaluasi potensi viralitas & retensi jika dipotong menjadi konten klip 30-60 detik (viral_score 1.0 - 10.0).
3. Buatkan 1 kalimat "Hook Idea" pembuka yang sangat menarik dan menghentikan scrolling (dalam Bahasa Indonesia).
4. Berikan 1-2 kalimat alasan clipping & rekomendasi angle/potongan terbaiknya.

Respond ONLY with a valid JSON object in this format:
{{
  "niche_score": <float 1.0 to 10.0>,
  "viral_score": <float 1.0 to 10.0>,
  "hook_idea": "<Kalimat pembuka hook 1 baris yang bikin orang penasaran dalam Bahasa Indonesia>",
  "clipping_reason": "<1-2 kalimat alasan kenapa video ini bagus di-clip untuk akun ini>"
}}
"""

    # Direct REST API call (fast, non-blocking, 10s timeout)
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={api_key}"
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.4, "maxOutputTokens": 400}
    }

    try:
        r = requests.post(url, json=payload, timeout=10)
        if r.status_code == 200:
            data_resp = r.json()
            raw_text = data_resp["candidates"][0]["content"]["parts"][0]["text"]
            clean_json = re.sub(r"^```json\s*|\s*```$", "", raw_text.strip(), flags=re.MULTILINE)
            data = json.loads(clean_json)
            return {
                "niche_score": float(data.get("niche_score", 7.0)),
                "viral_score": float(data.get("viral_score", 7.0)),
                "hook_idea": data.get("hook_idea", ""),
                "clipping_reason": data.get("clipping_reason", "")
            }
        else:
            log.warning(f"Gemini API returned status {r.status_code}: {r.text[:200]}")
    except Exception as e:
        log.warning(f"Gemini evaluation error for '{video.get('title')}': {e}. Using heuristics.")

    return heuristic_niche_eval(
        video.get("title", ""),
        video.get("description", ""),
        video.get("tags", []),
        video.get("vph", 0),
        niche_topics
    )

print(f"✅ AI clipping evaluator initialized with model '{GEMINI_MODEL}'.")

✅ AI clipping evaluator initialized with model 'gemini-flash-lite-latest'.


## Section 6: Account-Tagged Notifiers (Telegram + Discord)
Dispatches formatted alert cards clearly labeled with the target account:
1. **Telegram:** Rich HTML photo card with the video thumbnail, target account label (e.g. `📱 Target: Crypto & Macro Wealth (#CryptoClips)`), velocity badge, AI hook, and direct copy-paste URL.
2. **Account Routing:** Automatically sends to the specific account's `telegram_chat_id` if defined, or falls back to your global `TELEGRAM_CHAT_ID`.

In [36]:
# ==============================================================================
# SECTION 6: Multi-Channel Notifiers with Account Tagging
# ==============================================================================

def send_telegram_alert(video: dict, account: dict, bot_token: str, chat_id: str) -> bool:
    """Send photo alert card with HTML caption to Telegram."""
    target_chat = account.get("telegram_chat_id") or chat_id
    if not bot_token or not target_chat:
        return False

    title = video.get("title", "")
    channel = video.get("channel_name", "")
    url = video.get("url", "")
    vph = video.get("vph", 0)
    views = f"{video.get('views', 0):,}"
    age_hours = video.get("age_hours", 0)
    multiplier = video.get("multiplier", 1.0)
    niche_score = video.get("niche_score", "-")
    viral_score = video.get("viral_score", "-")
    hook = video.get("hook_idea", "")
    reason = video.get("clipping_reason", "")
    thumb = video.get("thumbnail_url", "")
    acc_name = account.get("account_name", "General")
    acc_tag = account.get("tag", "")

    caption = (
        f"🚨 <b>VIRAL CLIP OPPORTUNITY</b> 🚨\n\n"
        f"📱 <b>Target Account:</b> <code>{acc_name}</code> ({acc_tag})\n"
        f"🎬 <b><a href='{url}'>{title}</a></b>\n"
        f"👤 <b>Channel:</b> {channel}\n\n"
        f"⚡ <b>Momentum:</b> <code>{views}</code> views in <b>{age_hours}h</b>\n"
        f"🚀 <b>Velocity:</b> <code>{vph:,.0f} VPH</code> (<b>{multiplier}x</b> normal)\n"
        f"📊 <b>Scores:</b> Niche: <code>{niche_score}/10</code> | Viral: <code>{viral_score}/10</code>\n\n"
        f"🎣 <b>Hook Idea:</b>\n<i>\"{hook}\"</i>\n\n"
        f"💡 <b>Clipping Angle:</b>\n{reason}\n\n"
        f"🔗 <b>URL for Clipping:</b> <code>{url}</code>"
    )

    # 1. Try sending as photo with caption
    if thumb:
        photo_url = f"https://api.telegram.org/bot{bot_token}/sendPhoto"
        try:
            r = requests.post(photo_url, json={
                "chat_id": target_chat,
                "photo": thumb,
                "caption": caption[:1024],
                "parse_mode": "HTML"
            }, timeout=10)
            if r.status_code == 200:
                log.info(f"Telegram photo alert sent for: {title}")
                return True
        except Exception as e:
            log.warning(f"Telegram sendPhoto failed: {e}. Falling back to text message.")

    # 2. Text message fallback
    msg_url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
    try:
        r = requests.post(msg_url, json={
            "chat_id": target_chat,
            "text": caption[:4096],
            "parse_mode": "HTML",
            "disable_web_page_preview": False
        }, timeout=10)
        return r.status_code == 200
    except Exception as e:
        log.error(f"Telegram alert error: {e}")
        return False

def send_discord_alert(video: dict, account: dict, webhook_url: str) -> bool:
    """Send rich embed to Discord webhook (skipped if URL not provided)."""
    if not webhook_url:
        return False

    acc_name = account.get("account_name", "General")
    acc_tag = account.get("tag", "")

    payload = {
        "username": "Viral Clip Radar",
        "avatar_url": "https://cdn-icons-png.flaticon.com/512/1384/1384060.png",
        "embeds": [{
            "title": f"🔥 Viral Clip Alert: {video.get('title', '')[:200]}",
            "url": video.get("url", ""),
            "color": 16729856,
            "author": {
                "name": f"{acc_name} ({acc_tag})",
            },
            "thumbnail": {
                "url": video.get("thumbnail_url", "")
            },
            "fields": [
                {
                    "name": "🎬 Channel",
                    "value": video.get("channel_name", "YouTube"),
                    "inline": True
                },
                {
                    "name": "⚡ Velocity & Views",
                    "value": f"**{video.get('views', 0):,} views** in **{video.get('age_hours', 0)}h**\n`{video.get('vph', 0):,.0f} VPH` ({video.get('multiplier', 1.0)}x baseline)",
                    "inline": True
                },
                {
                    "name": "📊 AI Scores",
                    "value": f"Niche: `{video.get('niche_score', '-')}/10`\nViral: `{video.get('viral_score', '-')}/10`",
                    "inline": True
                },
                {
                    "name": "🎣 Suggested Clip Hook",
                    "value": f"*{video.get('hook_idea', 'N/A')}*",
                    "inline": False
                },
                {
                    "name": "💡 Why Clip This?",
                    "value": video.get("clipping_reason", "High momentum video.")[:1000],
                    "inline": False
                },
                {
                    "name": "📋 Video URL (Ready for Clipping)",
                    "value": f"`{video.get('url', '')}`",
                    "inline": False
                }
            ],
            "footer": {
                "text": f"Scout Agent • Target Account: {acc_name}"
            },
            "timestamp": datetime.now(timezone.utc).isoformat()
        }]
    }

    try:
        r = requests.post(webhook_url, json=payload, timeout=10)
        return r.status_code in (200, 204)
    except Exception as e:
        log.warning(f"Discord webhook error: {e}")
        return False

def dispatch_notifications(video: dict, account: dict):
    """Deliver alerts across configured channels."""
    sent_any = False
    if TELEGRAM_BOT_TOKEN:
        ok = send_telegram_alert(video, account, TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID)
        sent_any = sent_any or ok

    if DISCORD_WEBHOOK_URL:
        ok = send_discord_alert(video, account, DISCORD_WEBHOOK_URL)
        sent_any = sent_any or ok

    return sent_any

print("✅ Notification dispatchers configured with multi-account routing.")

✅ Notification dispatchers configured with multi-account routing.


## Section 7: Multi-Account Pipeline Orchestrator & Execution
Run this cell to execute the radar across all your clipping accounts:
1. Iterates through each profile in `CLIPPING_ACCOUNTS`.
2. Scans channels assigned to each niche.
3. Checks MongoDB (or local cache) for prior alerts in that specific niche.
4. Identifies viral spikes and runs account-specific Gemini evaluations.
5. Dispatches alerts tagged with the exact target account name.
6. Saves records into MongoDB for cloud persistence.
7. Prints a consolidated multi-account summary table.

In [40]:
# ==============================================================================
# SECTION 7: Multi-Account Pipeline Orchestrator
# ==============================================================================

def run_daily_scout(target_niche_id: Optional[str] = None):
    print("\n" + "="*75)
    print("🚀 STARTING MULTI-ACCOUNT VIRAL CLIP SCOUT RADAR")
    print("="*75)

    accounts_to_run = CLIPPING_ACCOUNTS
    if target_niche_id:
        accounts_to_run = [a for a in CLIPPING_ACCOUNTS if a.get("niche_id") == target_niche_id]
        if not accounts_to_run:
            print(f"❌ Niche ID '{target_niche_id}' not found in configuration.")
            return []

    overall_summary = []
    total_alerted = 0

    for account in accounts_to_run:
        acc_name = account.get("account_name", "General")
        niche_id = account.get("niche_id", "default")
        channels = account.get("channels", [])

        print(f"\n📂 [{acc_name}] (Niche: {niche_id}) — Scanning {len(channels)} channel(s)...")

        # 1. Gather latest video candidates for this account
        raw_candidates = []
        for ch in channels:
            videos = fetch_channel_videos(ch, limit=DETECTION_CONFIG["max_videos_per_channel"])
            unprocessed = [v for v in videos if not state_mgr.is_already_alerted(v["video_id"], niche_id)]
            raw_candidates.extend(unprocessed)
            print(f"   • {ch.get('name')}: {len(videos)} found, {len(unprocessed)} new for this account")
            time.sleep(0.2)

        if not raw_candidates:
            print(f"   ✅ All videos up-to-date for '{acc_name}'.")
            continue

        # 2. Enrich via YouTube Data API if key exists
        if YOUTUBE_API_KEY:
            candidates = enrich_via_youtube_api(raw_candidates, YOUTUBE_API_KEY)
        else:
            candidates = raw_candidates

        # 3. Analyze candidates for this account
        for v in candidates:
            metrics = calculate_video_metrics(v)
            is_spike = metrics["is_outlier"]
            title_snippet = metrics["title"][:38] + "..." if len(metrics["title"]) > 38 else metrics["title"]

            if is_spike:
                print(f"   ⚡ Momentum! Evaluating '{title_snippet}' for [{acc_name}]...")
                eval_res = evaluate_clippability_gemini(metrics, account, GEMINI_API_KEY)
                metrics.update(eval_res)

                passes_niche = metrics["niche_score"] >= DETECTION_CONFIG["min_niche_score"]
                passes_viral = metrics["viral_score"] >= DETECTION_CONFIG["min_viral_score"]

                if passes_niche and passes_viral:
                    print(f"      🎯 MATCH! Sending Alert to Telegram ({account.get('tag')})")
                    dispatch_notifications(metrics, account)
                    state_mgr.record_alert(metrics, account)
                    total_alerted += 1
                    status = "🚨 ALERTED"
                else:
                    status = f"Filtered (N:{metrics['niche_score']} V:{metrics['viral_score']})"
            else:
                status = "Normal Pace"
                metrics["niche_score"] = "-"
                metrics["viral_score"] = "-"

            overall_summary.append([
                acc_name[:16],
                metrics["channel_name"][:16],
                title_snippet,
                f"{metrics['age_hours']}h",
                f"{metrics['views']:,}",
                f"{metrics['vph']:,.0f}",
                f"{metrics['multiplier']}x",
                f"{metrics['niche_score']}",
                f"{metrics['viral_score']}",
                status
            ])

    # Print Combined Report Table
    headers = ["Target Account", "Channel", "Title", "Age", "Views", "VPH", "Mult", "Niche", "Viral", "Outcome"]
    print("\n" + tabulate(overall_summary, headers=headers, tablefmt="fancy_grid"))

    print("\n" + "="*75)
    print(f"🎉 RADAR COMPLETE: {total_alerted} new clip opportunit{'ies' if total_alerted != 1 else 'y'} dispatched across {len(accounts_to_run)} accounts!")
    print("="*75)
    return overall_summary

# Execute radar for all clipping accounts
report = run_daily_scout()


🚀 STARTING MULTI-ACCOUNT VIRAL CLIP SCOUT RADAR

📂 [Crypto, Trading & Investing] (Niche: crypto_trading) — Scanning 3 channel(s)...
   • Astronacci International: 5 found, 5 new for this account
   • Timothy Ronald: 5 found, 5 new for this account
   • Raymond Chin: 5 found, 5 new for this account
   ⚡ Momentum! Evaluating 'Update Rupiah!! Mengarah ke Rp18,800-1...' for [Crypto, Trading & Investing]...
      🎯 MATCH! Sending Alert to Telegram (#CryptoTradingID)
   ⚡ Momentum! Evaluating 'Begini cara membaca market yang benar ...' for [Crypto, Trading & Investing]...


   ⚡ Momentum! Evaluating 'Sejauh apa bitcoin akan turun?' for [Crypto, Trading & Investing]...


   ⚡ Momentum! Evaluating 'Realita Pahit Buat Lo yang Mau Sukses ...' for [Crypto, Trading & Investing]...


   ⚡ Momentum! Evaluating 'AI Mulai Keliatan Mirip Terminator' for [Crypto, Trading & Investing]...
   ⚡ Momentum! Evaluating 'Mundurnya Gibran Sebenernya Bikin Damp...' for [Crypto, Trading & Investing]...


   ⚡ Momentum! Evaluating 'Siapa Yang Bakal Gantiin Gibran?' for [Crypto, Trading & Investing]...
   ⚡ Momentum! Evaluating 'Di Balik Ancaman Pemakzulan Gibran: Pr...' for [Crypto, Trading & Investing]...


   ⚡ Momentum! Evaluating 'Bansos Ga Tepat Sasaran?!' for [Crypto, Trading & Investing]...

📂 [Bisnis, Startup & Entrepreneurship] (Niche: business_startups) — Scanning 4 channel(s)...
   • Kasisolusi: 5 found, 5 new for this account
   • Leon Hartono (The Overpost): 5 found, 5 new for this account
   • Raymond Chin: 5 found, 5 new for this account
   • Deddy Corbuzier: 5 found, 5 new for this account
   ⚡ Momentum! Evaluating 'Ternyata Ini yang Bikin Lo Jerawatan‼️...' for [Bisnis, Startup & Entrepreneurship]...


... (10 outputs left out)

   ⚡ Momentum! Evaluating 'Umur.' for [Self-Growth, Habits & Karir]...


   ⚡ Momentum! Evaluating 'Ketika Dokter Tirta, Aloy, dan Niko Be...' for [Self-Growth, Habits & Karir]...


   ⚡ Momentum! Evaluating 'Strategi dr. Tirta.' for [Self-Growth, Habits & Karir]...



╒══════════════════╤══════════════════╤════════════════════════════════════════════╤════════╤═══════════╤════════╤════════╤═════════╤═════════╤════════════════════════╕
│ Target Account   │ Channel          │ Title                                      │ Age    │ Views     │ VPH    │ Mult   │ Niche   │ Viral   │ Outcome                │
╞══════════════════╪══════════════════╪════════════════════════════════════════════╪════════╪═══════════╪════════╪════════╪═════════╪═════════╪════════════════════════╡
│ Crypto, Trading  │ Astronacci Inter │ GOLD MAU KEMANA SIH? Kenapa Susah bang...  │ 0.4h   │ 972       │ 2,320  │ 2.23x  │ -       │ -       │ Normal Pace            │
├──────────────────┼──────────────────┼────────────────────────────────────────────┼────────┼───────────┼────────┼────────┼─────────┼─────────┼────────────────────────┤
│ Crypto, Trading  │ Astronacci Inter │ Update Rupiah!! Mengarah ke Rp18,800-1...  │ 27.8h  │ 341,575   │ 12,269 │ 11.78x │ 9.8     │ 9.6     │ 🚨 ALERTED 